<a href = "https://www.pieriantraining.com"><img src="../PT Centered Purple.png"> </a>

<em style="text-align:center">Copyrighted by Pierian Training</em>

# Model IO Exercise 

The purpose of this exercise is to test your understanding of building out Model IO systems. You will also hopefully notice the need to chain responses together, which we will cover later in this course!

Watch the video for a full overview on minimum outputs this class should be capable of, but feel free to expand on this project, or to just treat it as a code-along!

## History Quiz

Our main goal is to use LangChain and Python to create a very simple class with a few methods for:
* Writing a historical question that has a date as the correct answer
* Getting the correct answer from LLM
* Getting a Human user's best guess at at correct answer
* Checking/reporting the difference between the correct answer and the user answer

### Suggested Imports

Feel free to accomplish this task however you prefer!

In [ ]:
import os


In [16]:
from langchain.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from datetime import datetime
from langchain.output_parsers import DatetimeOutputParser
from langchain_openai import ChatOpenAI

model = ChatOpenAI()


In [20]:
class HistoryQuiz():
    
    def create_history_question(self,topic):
        '''
        This method should output a historical question about the topic that has a date as the correct answer.
        For example:
        
            "On what date did World War 2 end?"
            
        '''
        
        system_message_prompt_template = SystemMessagePromptTemplate.from_template("Your task is to respond with a historical question about provided topic that has a date as the correct answer. For example: On what date did the World War 2 end?")

        human_message_prompt_template = HumanMessagePromptTemplate.from_template("{topic}")

        chat_prompt_template = ChatPromptTemplate.from_messages([
            system_message_prompt_template,
            human_message_prompt_template,
        ])

        request = chat_prompt_template.format_prompt(topic=topic).to_messages()
        question = model(request)
       
        return question
    
    def get_AI_answer(self,question):
        '''
        This method should get the answer to the historical question from the method above.
        Note: This answer must be in datetime format! Use DateTimeOutputParser to confirm!
        
        September 2, 1945 --> datetime.datetime(1945, 9, 2, 0, 0)
        '''

        system_message_prompt_template = SystemMessagePromptTemplate.from_template(
            "Your task is to answer the historical question only in datetime format that python can understand and parse.\n{format_instructions}"
        )

        human_message_prompt_template = HumanMessagePromptTemplate.from_template("{question}")

        output_parser = DatetimeOutputParser()

        chat_prompt_template = ChatPromptTemplate.from_messages([
            system_message_prompt_template,
            human_message_prompt_template,
        ])

        request = chat_prompt_template.format_prompt(
            question=question, 
            format_instructions=output_parser.get_format_instructions()).to_messages()
        

        result = model(request)

        correct_datetime = output_parser.parse(result.content)
        
        return correct_datetime
    
    def get_user_answer(self,question):
        '''
        This method should grab a user answer and convert it to datetime. It should collect a Year, Month, and Day.
        You can just use input() for this.
        '''
        print(question)
        year = int(input("Enter the year:"))
        month = int(input("Enter the month (1-12):"))
        day = int(input("Enter the day (1-31):"))

        user_datetime = datetime(year, month, day)
        
        return user_datetime
        
        
    def check_user_answer(self,user_answer:datetime,ai_answer:datetime):
        '''
        Should check the user answer against the AI answer and return the difference between them
        '''
        # print or return the difference between the answers here!
        print("The difference between the dates is:", ai_answer-user_answer)
        

### Example Usage

Feel free to expand or edit this project. To keep things simple we have every method return an object that will then feed into a new method!

In [21]:
quiz_bot = HistoryQuiz()

In [22]:
question = quiz_bot.create_history_question(topic='World War 2')

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [37]:
question

'On what date did the United States officially enter World War II?'

In [38]:
ai_answer = quiz_bot.get_AI_answer(question)

In [39]:
# Day After Pearl Harbor
ai_answer

datetime.datetime(1941, 12, 8, 0, 0)

In [40]:
user_answer = quiz_bot.get_user_answer(question)

On what date did the United States officially enter World War II?


Enter the year:  1941
Enter the month (1-12):  12
Enter the day (1-31):  1


In [41]:
user_answer

datetime.datetime(1941, 12, 1, 0, 0)

In [42]:
quiz_bot.check_user_answer(user_answer,ai_answer)

The difference between the dates is: -7 days, 0:00:00
